# SECTION 1: IMPORTS AND SETUP

In [2]:
pip install numpy pandas matplotlib nltk scikit-learn scipy

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: C:\Users\quratulain.adnan\AppData\Local\Programs\Python\Python314\python.exe -m pip install --upgrade pip


In [3]:
import numpy as np
import pandas as pd
import re
import string
import nltk

import matplotlib.pyplot as plt

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.cluster import KMeans

# Evaluation
from sklearn.metrics import (
    f1_score,
    confusion_matrix,
    adjusted_rand_score,
    normalized_mutual_info_score
)

# Download NLTK resources
nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('omw-1.4')
nltk.download('averaged_perceptron_tagger')
nltk.download('averaged_perceptron_tagger_eng')

# Reproducibility
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

print("Setup complete.")

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\quratulain.adnan\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\quratulain.adnan\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\quratulain.adnan\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\quratulain.adnan\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     C:\Users\quratulain.adnan\AppData\Roaming\nltk_data...


Setup complete.


[nltk_data]   Package omw-1.4 is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     C:\Users\quratulain.adnan\AppData\Roaming\nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     C:\Users\quratulain.adnan\AppData\Roaming\nltk_data...
[nltk_data]   Package averaged_perceptron_tagger_eng is already up-to-
[nltk_data]       date!


# Section 2: Dataset Loading

In [4]:
from sklearn.datasets import fetch_20newsgroups

# using 6 categories ==> WSD is computationally expensive.
#   - enough diversity for interesting clustering
#   - Small enough for WSD/lexical chains to run in reasonable time
CATEGORIES = [
    'rec.sport.hockey',
    'sci.space',
    'talk.politics.guns',
    'comp.graphics',
    'rec.autos',
    'sci.med',
]

# Number of clusters = number of categories we chose
# k-Means will find exactly this many clusters
NUM_CLUSTERS = len(CATEGORIES)   # = 6

# subset='all'  → use both train and test splits combined
# remove=...    → strip headers, footers, and quoted replies
print("Loading 20 Newsgroups dataset...")
newsgroups = fetch_20newsgroups(
    subset='all',
    categories=CATEGORIES,
    remove=('headers', 'footers', 'quotes'),
    random_state=42          
)

# newsgroups.data  → list of raw document strings
# newsgroups.target → array of integer class labels (0, 1, 2, ...)
# newsgroups.target_names → the human-readable category names

# WSD is slow. limit each category to MAX_PER_CLASS documents.
# keeps the dataset balanced AND fast.
import numpy as np

MAX_PER_CLASS = 100

np.random.seed(42)
selected_indices = []
for class_id in range(NUM_CLUSTERS):
    # find all indices belonging to this class
    class_indices = np.where(newsgroups.target == class_id)[0]
    # Randomly pick at most MAX_PER_CLASS of them
    
    chosen = np.random.choice(
        class_indices,
        size=min(MAX_PER_CLASS, len(class_indices)),
        replace=False   # no duplicates
    )
    selected_indices.extend(chosen.tolist())   # will contain around 600 docs

# shuffle the selected indices so classes aren't in blocks
np.random.shuffle(selected_indices)

# final document list and label list 
documents   = [newsgroups.data[i]   for i in selected_indices]
true_labels = [newsgroups.target[i] for i in selected_indices]

print(f"\nCategories selected : {CATEGORIES}")
print(f"Total documents     : {len(documents)}")
print(f"Total labels        : {len(true_labels)}")
print(f"Documents per class : {MAX_PER_CLASS} (max)")
print(f"\nSample document (first 300 chars):\n")
print(documents[0][:300])
print("\nCorresponding label:", newsgroups.target_names[true_labels[0]])

Loading 20 Newsgroups dataset...

Categories selected : ['rec.sport.hockey', 'sci.space', 'talk.politics.guns', 'comp.graphics', 'rec.autos', 'sci.med']
Total documents     : 600
Total labels        : 600
Documents per class : 100 (max)

Sample document (first 300 chars):


Here goes:

More than a few years back (if you were born that year, you can legally drink),
we tried it out.  We found an 8 ft. deep cistern that we lined with some 10 ft.
2X6s.  We put a large can (one of those industrial sized pork'n beans cans)
stuffed with oily rags and scraps of wood in the bo

Corresponding label: talk.politics.guns


# SECTION 3: PREPROCESSING 

In [5]:
import re
import nltk
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
from nltk.tokenize import word_tokenize

# nltk data files already downloaded in Section 1

# initialise tools we will be reusing across all steps
stemmer        = PorterStemmer()          
STOPWORDS      = set(stopwords.words('english'))  # common English words to remove

# After stemming, words become unrecognisable (e.g. "running" → "run").
# WordNet needs real English words, not stems !!
# So we keep a global dictionary that maps each stem back to the most common original word — used in Steps 4 and 5.
stem_to_word = {}   # e.g.  {"run": "running",  "space": "space"}

# -----------------------------------------------------------------------------
def clean_text(text):
    """
    Remove noise from a raw document string.
      1. Lowercase everything
      2. Remove URLs
      3. Remove email addresses
      4. Remove non-alphabetic characters (numbers, punctuation, etc.)
      5. Collapse multiple spaces into one
    """
    text = text.lower()                            # 1. lowercase
    text = re.sub(r'http\S+|www\.\S+', ' ', text) # 2. remove URLs
    text = re.sub(r'\S+@\S+', ' ', text)          # 3. remove emails
    text = re.sub(r'[^a-z\s]', ' ', text)         # 4. keep only a-z and spaces
    text = re.sub(r'\s+', ' ', text).strip()       # 5. collapse whitespace
    return text


# -----------------------------------------------------------------------------
def tokenize_and_filter(text):
    """
      1. Tokenize
      2. Remove stopwords
      3. Remove very short words (len < 3)
      4. Remove very long words (len > 20)
    """

    tokens = word_tokenize(text)                   # 1. split into words
    tokens = [t for t in tokens
              if t not in STOPWORDS                # 2. drop stopwords
              and 3 <= len(t) <= 20]               # 3 & 4. length filter
    return tokens


# -----------------------------------------------------------------------------
def stem_tokens(tokens):
    
    # Reduce each token to its root (stem) 
    # updates the global stem_to_word mapping so we can recover real words for WordNet lookups later.
     
    stemmed = []
    for word in tokens:
        stem = stemmer.stem(word)        # e.g. "running" → "run"
        stemmed.append(stem)

        # Stores the shortest real-word form for each stem
        # (shorter originals tend to be the base form)
        if stem not in stem_to_word:
            stem_to_word[stem] = word
        else:
            # prefer the shorter word as the representative
            if len(word) < len(stem_to_word[stem]):
                stem_to_word[stem] = word

    return stemmed


# -----------------------------------------------------------------------------
def preprocess_document(text):
 # clean_text → tokenize_and_filter → stem_tokens
    text   = clean_text(text)            
    tokens = tokenize_and_filter(text)   
    stems  = stem_tokens(tokens)         
    return stems


# -----------------------------------------------------------------------------
def preprocess_corpus(documents):
    """
    Apply preprocess_document to every document in the corpus.
    """
    preprocessed = []
    total = len(documents)

    for i, doc in enumerate(documents):
        # Progress update every 100 documents
        if (i + 1) % 100 == 0 or (i + 1) == total:
            print(f"  Preprocessing document {i+1}/{total}...")
        preprocessed.append(preprocess_document(doc))

    return preprocessed


# preprocessing on the full corpus
print("Starting preprocessing...\n")
preprocessed_docs = preprocess_corpus(documents)
print("\nPreprocessing complete.")

print(f"\nTotal documents preprocessed : {len(preprocessed_docs)}")
print(f"Stem-to-word mapping size    : {len(stem_to_word)} stems")
print(f"\nExample — original (first 200 chars):")
print(documents[0][:200])
print(f"\nExample — preprocessed tokens (first 20):")
print(preprocessed_docs[0][:20])

Starting preprocessing...

  Preprocessing document 100/600...
  Preprocessing document 200/600...
  Preprocessing document 300/600...
  Preprocessing document 400/600...
  Preprocessing document 500/600...
  Preprocessing document 600/600...

Preprocessing complete.

Total documents preprocessed : 600
Stem-to-word mapping size    : 7609 stems

Example — original (first 200 chars):

Here goes:

More than a few years back (if you were born that year, you can legally drink),
we tried it out.  We found an 8 ft. deep cistern that we lined with some 10 ft.
2X6s.  We put a large can (

Example — preprocessed tokens (first 20):
['goe', 'year', 'back', 'born', 'year', 'legal', 'drink', 'tri', 'found', 'deep', 'cistern', 'line', 'put', 'larg', 'one', 'industri', 'size', 'pork', 'bean', 'can']


# SECTION 4: NOUN EXTRACTION FUNCTIONS

In [7]:
from nltk import pos_tag

# NLTK's pos_tag assigns a Part-of-Speech tag to each word
# Tags that start with 'NN' are nouns:
#   NN  = singular noun        e.g. "rocket"
#   NNS = plural noun          e.g. "rockets"
#   NNP = proper noun singular e.g. "NASA"
#   NNPS= proper noun plural   e.g. "Americans"
NOUN_TAGS = {'NN', 'NNS', 'NNP', 'NNPS'}


def extract_nouns_from_doc(stem_list):
    
    # run POS tagging and return only the (stem, real_word) pairs that are nouns.
    # The REAL WORD is needed for WordNet lookups
    

    # recover real words from stems using the global mapping
    real_words = [stem_to_word.get(stem, stem) for stem in stem_list]
    # we just use the stem itself as the word, if no mapping

    # returns a list of (word, tag) tuples.
    tagged = pos_tag(real_words)

    # keep tokens whose POS tag indicates a noun
    # zip stems and tagged results together so we can return both
    noun_pairs = []
    for stem, (word, tag) in zip(stem_list, tagged):
        if tag in NOUN_TAGS:
            noun_pairs.append((stem, word))

    return noun_pairs


# -----------------------------------------------------------------------------
def extract_nouns_from_corpus(preprocessed_docs):
    # Input  : list of stem lists (full preprocessed corpus)
    # Output : list of noun-pair lists (one list of tuples per document)
    noun_docs = []
    total = len(preprocessed_docs)

    for i, stem_list in enumerate(preprocessed_docs):
        if (i + 1) % 100 == 0 or (i + 1) == total:
            print(f"  Extracting nouns from document {i+1}/{total}...")
        noun_pairs = extract_nouns_from_doc(stem_list)
        noun_docs.append(noun_pairs)

    return noun_docs


print("Starting noun extraction...\n")
noun_docs = extract_nouns_from_corpus(preprocessed_docs)
print("\nNoun extraction complete.")

print(f"\nTotal documents processed : {len(noun_docs)}")

# Show stats: how many tokens survive noun filtering on average
total_stems = sum(len(doc) for doc in preprocessed_docs)
total_nouns = sum(len(doc) for doc in noun_docs)
print(f"Total stems (all tokens)  : {total_stems}")
print(f"Total nouns kept          : {total_nouns}")
print(f"Noun retention rate       : {100 * total_nouns / total_stems:.1f}%")

# Show example for first document
print(f"\nExample — stems (first 15):")
print(preprocessed_docs[0][:100])
print(f"\nExample — noun (stem, word) pairs (first 10):")
print(noun_docs[0][:100])

Starting noun extraction...

  Extracting nouns from document 100/600...
  Extracting nouns from document 200/600...
  Extracting nouns from document 300/600...
  Extracting nouns from document 400/600...
  Extracting nouns from document 500/600...
  Extracting nouns from document 600/600...

Noun extraction complete.

Total documents processed : 600
Total stems (all tokens)  : 43240
Total nouns kept          : 23216
Noun retention rate       : 53.7%

Example — stems (first 15):
['goe', 'year', 'back', 'born', 'year', 'legal', 'drink', 'tri', 'found', 'deep', 'cistern', 'line', 'put', 'larg', 'one', 'industri', 'size', 'pork', 'bean', 'can', 'stuf', 'oili', 'rag', 'scrap', 'wood', 'bottom', 'light', 'fire', 'lower', 'box', 'spc', 'swc', 'heard', 'pop', 'one', 'solid', 'bang', 'sever', 'fizzzz', 'shussss', 'thought', 'excit', 'boldli', 'climb', 'find', 'none', 'bullet', 'left', 'sever', 'shell', 'lie', 'around', 'bottom', 'well', 'board', 'die', 'smoke', 'inhal', 'shell', 'still', 'live